In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import numpy as np
import pandas as pd
from PIL import Image
import os

In [2]:
base_image_dir = "/home/jhong.dongyou/Desktop/orange_smile/data/tanlikesmath/diabetic-retinopathy-resized/versions/7"
df = pd.read_csv(os.path.join(base_image_dir, 'trainLabels_cropped.csv'))
df['path'] = df['image'].map(lambda x: os.path.join(base_image_dir,'resized_train_cropped/resized_train_cropped','{}.jpeg'.format(x)))
df = df.drop(columns=['image'])
df = df.sample(frac=1).reset_index(drop=True) #shuffle dataframe
df['level'] = df['level'].astype(int) # Disease or no disease
df.head(10)

,Unnamed: 0.1,Unnamed: 0,level,path
0,30837,30853,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
1,26405,26420,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
2,9244,9251,2,/home/jhong.dongyou/Desktop/orange_smile/data/...
3,2610,2613,2,/home/jhong.dongyou/Desktop/orange_smile/data/...
4,4414,4418,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
5,28808,28824,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
6,12465,12474,2,/home/jhong.dongyou/Desktop/orange_smile/data/...
7,12587,12596,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
8,26557,26572,0,/home/jhong.dongyou/Desktop/orange_smile/data/...
9,29683,29699,0,/home/jhong.dongyou/Desktop/orange_smile/data/...


In [3]:
from torch.utils.data import Dataset
from PIL import Image

class DiabeticRetinopathyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['level']
        
        image = Image.open(img_path).convert("RGB") 
        image = image.resize((224, 224))
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

    # Load dataset
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
from sklearn.model_selection import train_test_split
train_dataset, val_dataset = train_test_split(df, test_size=0.2)

train_dataset = DiabeticRetinopathyDataset(train_dataset, transform=transform)
val_dataset = DiabeticRetinopathyDataset(val_dataset, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

In [5]:
# Define model
model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)  # 0 to 4, 5 classes

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

/blue/ruogu.fang/jhong.dongyou/.conda/envs/resnet50/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/blue/ruogu.fang/jhong.dongyou/.conda/envs/resnet50/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
for batch in train_loader:
    print("Batch loaded!")
    break  # load one batch，make sure DataLoader works


Batch loaded!


In [7]:
from tqdm import tqdm
# Training loop
epochs = 100


for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Training Epoch {epoch+1}")
    for i, (inputs, labels) in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        # update bar
        progress_bar.set_postfix(loss=running_loss / (i + 1))

    print(f"Epoch {epoch+1} completed, Average Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        y_pred.extend(probs)   # shape: (N_samples, 5)
        y_true.extend(labels.cpu().numpy())

Epoch 1/100


Training Epoch 1: 100%|██████████| 1756/1756 [07:23<00:00,  3.96it/s, loss=0.722]


Epoch 1 completed, Average Loss: 0.7215
Epoch 2/100


Training Epoch 2: 100%|██████████| 1756/1756 [07:18<00:00,  4.01it/s, loss=0.658]


Epoch 2 completed, Average Loss: 0.6579
Epoch 3/100


Training Epoch 3: 100%|██████████| 1756/1756 [06:55<00:00,  4.23it/s, loss=0.636]


... (191 outputs left out)

Training Epoch 99: 100%|██████████| 1756/1756 [07:00<00:00,  4.17it/s, loss=0.125]


Epoch 99 completed, Average Loss: 0.1247
Epoch 100/100


Training Epoch 100: 100%|██████████| 1756/1756 [07:03<00:00,  4.15it/s, loss=0.123]


Epoch 100 completed, Average Loss: 0.1229


In [8]:
# Save model in torchvision-compatible format
torch.save(model.state_dict(), 'resnet50_DR_5class.pth')

In [9]:
# Compute AUC for multi-class
auc = roc_auc_score(y_true, y_pred, multi_class='ovr')
print("AUC:", auc)

AUC: 0.8529521153535307
